# E-commerce Data Preprocessing | Spark

In [1]:
import pyspark
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("E-commerce Data Preprocessing") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

auto_df = spark.read.format("csv").option("header", True).load("hdfs://namenode:9000/project-data/ecommerce.csv")
auto_df.show()

+--------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|          event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+--------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 00:00:...|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 00:00:...|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 00:00:...|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 00:00:...|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 00:00:...|      view|   1004237|205301355563188265

In [2]:
# === explore some metadata ===
# num rows
print("=== num rows ===")
print(auto_df.count())

# data schema
print("=== data schema ===")
auto_df.printSchema()

# count null
from pyspark.sql.functions import col, sum, when
null_counts = auto_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in auto_df.columns
])

print("=== nulls count ===")
null_counts.show()

=== num rows ===
15000000
=== data schema ===
root
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_session: string (nullable = true)

=== nulls count ===
+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|  brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+
|         0|         0|         0|          0|      4969146|2130438|    0|      0|           1|
+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+



In [5]:
# === reducing data ===
# removing duplicates
before = auto_df.count()
print(f"before: {before}")

df = auto_df.dropDuplicates()

after = df.count()
print(f"after: {after}")
print("no duplicates" if before == after else f"{before - after} duplicates removed")

before: 15000000
after: 14991162
8838 duplicates removed


In [6]:
# removing rows with null values in brand and category
before = df.count()
print(f"before: {before}")

df = df.dropna()

after = df.count()
print(f"after: {after}")
print("no nulls" if before == after else f"{before - after} records dropped")

before: 14991162
after: 9227911
5763251 records dropped


In [7]:
# checking event_type values
df.groupBy("event_type").count().show()

+----------+-------+
|event_type|  count|
+----------+-------+
|  purchase| 189592|
|      view|8802892|
|      cart| 235427|
+----------+-------+



In [8]:
# we don't need all the records of event_type "view"
# we can take a sample of it instead
views = df.filter(col("event_type") == "view")
others = df.filter(col("event_type") != "view")

views_sampled = views.sample(fraction=0.2, seed=42)  # keep 20%

df = others.union(views_sampled)
df.groupBy("event_type").count().show()
print(f"rows after: {df.count()}")

+----------+-------+
|event_type|  count|
+----------+-------+
|  purchase| 189592|
|      cart| 235427|
|      view|1761443|
+----------+-------+

rows after: 2186462


In [9]:
# casting price to double
df = df.withColumn("price", col("price").cast("double"))
df.printSchema()

root
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_session: string (nullable = true)



In [10]:
# normalize strings
from pyspark.sql.functions import trim, lower
df = df.withColumn("category_code", trim(lower(col("category_code"))))

In [11]:
# creating columns for product name and category
from pyspark.sql.functions import split
df = df.withColumn("product_name", split(col("category_code"), "\\.").getItem(1))
df = df.withColumn("category", split(col("category_code"), "\\.").getItem(0))

df = df.drop("category_code")
df = df.drop("category_id")
df.show()

+--------------------+----------+----------+---------+-------+---------+--------------------+------------+-----------+
|          event_time|event_type|product_id|    brand|  price|  user_id|        user_session|product_name|   category|
+--------------------+----------+----------+---------+-------+---------+--------------------+------------+-----------+
|2019-10-01 02:39:...|  purchase|   1004670|  samsung|1286.75|518794940|434f148f-e292-402...|  smartphone|electronics|
|2019-10-01 02:53:...|  purchase|   1002544|    apple| 464.13|554090147|b440a0b8-397b-446...|  smartphone|electronics|
|2019-10-01 02:58:...|      cart|   1004321|   huawei| 243.25|555467501|19185d47-b7fd-424...|  smartphone|electronics|
|2019-10-01 03:09:...|      cart|   1004741|   xiaomi| 185.71|546747843|58a601d3-cb81-429...|  smartphone|electronics|
|2019-10-01 03:17:...|  purchase|   1002544|    apple| 464.08|554090147|c5633b76-2109-40a...|  smartphone|electronics|
|2019-10-01 03:22:...|  purchase|   1002524|    

In [12]:
# checking values of price
df.describe(["price"]).show()

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|           2186462|
|   mean| 359.9316769694191|
| stddev|384.80249292880075|
|    min|              0.88|
|    max|           2574.07|
+-------+------------------+



In [20]:
# no abnormal values or outliers

In [13]:
from pyspark.sql.functions import hour, dayofweek, month, year
df = df.withColumn("event_time", col("event_time").cast("timestamp"))
df = df.withColumn("hour", hour(col("event_time")))
df = df.withColumn("day", dayofweek(col("event_time"))) # 1 --> Sunday
df = df.withColumn("month", month(col("event_time")))
df = df.withColumn("year", year(col("event_time")))

df = df.drop("event_time")

# event_time was hard to deal with

df.show()

+----------+----------+---------+-------+---------+--------------------+------------+-----------+----+---+-----+----+
|event_type|product_id|    brand|  price|  user_id|        user_session|product_name|   category|hour|day|month|year|
+----------+----------+---------+-------+---------+--------------------+------------+-----------+----+---+-----+----+
|  purchase|   1004670|  samsung|1286.75|518794940|434f148f-e292-402...|  smartphone|electronics|   2|  3|   10|2019|
|  purchase|   1002544|    apple| 464.13|554090147|b440a0b8-397b-446...|  smartphone|electronics|   2|  3|   10|2019|
|      cart|   1004321|   huawei| 243.25|555467501|19185d47-b7fd-424...|  smartphone|electronics|   2|  3|   10|2019|
|      cart|   1004741|   xiaomi| 185.71|546747843|58a601d3-cb81-429...|  smartphone|electronics|   3|  3|   10|2019|
|  purchase|   1002544|    apple| 464.08|554090147|c5633b76-2109-40a...|  smartphone|electronics|   3|  3|   10|2019|
|  purchase|   1002524|    apple| 514.76|542388547|95126

In [ ]:
# end of preprocessing
df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("hdfs://namenode:9000/project-data/preprocessed-data")

In [15]:
spark.stop()